<a href="https://colab.research.google.com/github/ahhill27/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()

        except requests.RequestException:
            raise

        if attempt < tries - 1:
            time.sleep(2 ** attempt)

    raise RuntimeError('Weather request failed after', tries, 'attempts')
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
used_fallback = False

try:
    weather = get_weather('2026-09-05', '2026-10-03')

    weather['time'] = pd.to_datetime(weather['time'])

    dates = pd.to_datetime(['2026-09-05', '2026-10-03'])
    weather = weather[weather['time'].isin(dates)].reset_index(drop=True)

except Exception as e:
    print(f"Weather API failed: {e}")
    used_fallback = True

print("Used fallback:", used_fallback)

Used fallback: False


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'

print(wx)

        time  temperature_2m_max  precipitation_sum      source
0 2026-09-05                29.4                2.5  open-meteo
1 2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
    'source': 'source'
})

wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
sept = wx.loc[wx['date'] == '2026-09-05', 'precip_mm'].iloc[0]
oct = wx.loc[wx['date'] == '2026-10-03', 'precip_mm'].iloc[0]

wettest = wx.loc[wx['precip_mm'].idxmax()]
print(wettest['date'], 'was the wettest day with', wettest['precip_mm'], 'mm of rain.')
print(wettest['date'], 'had', oct - sept, 'mm more rain than', wx['date'][0])

2026-10-03 00:00:00 was the wettest day with 14.7 mm of rain.
2026-10-03 00:00:00 had 12.2 mm more rain than 2026-09-05 00:00:00


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

coordinates = '38.03, -78.51'

print('Endpoint:', BASE)
print('Coordinates:', coordinates)
print("Date range:", wx['date'].min().date(), "to", wx['date'].max().date())
print('Pulled at:', datetime.now().strftime('%Y-%m-%d %H:%M'))

if wx['source'].eq('fallback sample').any():
    print("Data source: Fallback sample values")
else:
    print("Data source: Open-Meteo Archive API (real data)")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-11 04:17
Data source: Open-Meteo Archive API (real data)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If we merged get_eatehr onto a chart of sales, the merge would still run but all of the weather values will display as NaN. Becuase rain isnt displayed, we would not be able to show any relationship.

Since the code would still run, there is a chance that nobody would catch the issue. Without the error message, one would need to recognize the possible inconsistencies in the data and look into the cause of the issue. If they dont do this, the issue may continue to go unnoticed.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
charlottesville = get_weather('2026-09-05', '2026-10-03',
                              lat=38.03, lon=-78.51)

orlando = get_weather('2026-09-05', '2026-10-03',
                      lat=28.54, lon=-81.38)

def clean_weather(df, city):
    df = df.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    df['date'] = pd.to_datetime(df['date'])
    df = df[['date', 'temp_max', 'precip_mm']].copy()
    df['city'] = city
    return df[df['date'].isin(pd.to_datetime(
        ['2026-09-05', '2026-10-03']
    ))]

wx_both = pd.concat([
    clean_weather(charlottesville, 'Charlottesville'),
    clean_weather(orlando, 'Orlando')
], ignore_index=True)

print(wx_both)

        date  temp_max  precip_mm             city
0 2026-09-05      29.4        2.5  Charlottesville
1 2026-10-03      22.0       14.7  Charlottesville
2 2026-09-05      32.1        3.9          Orlando
3 2026-10-03      30.4       14.5          Orlando
